# FINE-TUNE lane — CLAMP on a public robust base (Colab, Track B / explore)

Fine-tune a public ℓ∞-AT base (**`Sehwag2021Proxy_R18`**, RobustBench) with the CLAMP term vs a
matched no-term control — does the term add on an already-strong base? **Pre-registered:** report
regardless. Extension — does NOT touch the paper. **Track A first**; run this in a spare session.

**Gate already passed (B2, in `eval_colab`):** clean @10k = 0.8459 (exact), convention `[0,1]`.

Matched pair, identical except the term (reuses `finetune_msd_clamp.py --base robustbench:...`):
`M1_ft` (fine-tune + CLAMP glue+scaffold) vs `M0_ft` (fine-tune, no term). ~8 ep, val-select
parity, input [0,1], eps triple 8/255·0.5·12, no double-normalize. Tiered read: **no-Square @1k**
first (fast Δ) → if promising, full 12-AA @10k.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')

## Setup — Drive · code zip · CIFAR · robustbench

In [ ]:
import hashlib, tarfile, zipfile, os, subprocess as sp, sys, importlib.util
DRIVE='/content/drive/MyDrive/attackdro'; REPO='/content/attackdro'
CODE_ZIP=f'{DRIVE}/attackdro_code.zip'; CIFAR=f'{DRIVE}/cifar-10-python.tar.gz'
OUT=f'{DRIVE}/explore3_ftinf'                 # Drive-direct outputs
os.makedirs(REPO, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP) as z: z.extractall(REPO)
assert os.path.exists(f'{REPO}/scripts/dev/finetune_msd_clamp.py'), 'stale zip — re-upload attackdro_code.zip'
os.makedirs(f'{REPO}/data', exist_ok=True)
if not os.path.exists(f'{REPO}/data/cifar-10-batches-py/data_batch_1'):
    if os.path.exists(CIFAR):
        assert hashlib.sha256(open(CIFAR,'rb').read()).hexdigest().startswith('6d958be074577803')
        with tarfile.open(CIFAR) as t: t.extractall(f'{REPO}/data')
    else:
        import torchvision
        torchvision.datasets.CIFAR10(f'{REPO}/data', train=True, download=True)
        torchvision.datasets.CIFAR10(f'{REPO}/data', train=False, download=True)
if importlib.util.find_spec('robustbench') is None: sp.run([sys.executable,'-m','pip','install','-q','robustbench'])
sp.run([sys.executable,'-m','pip','install','-q','-U','gdown'])                # clicks the gdrive token
if importlib.util.find_spec('autoattack') is None: sp.run([sys.executable,'-m','pip','install','-q','git+https://github.com/fra31/auto-attack.git'])
import torch
print('setup OK |', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU', '| out ->', OUT)

## Config

In [ ]:
BASE='robustbench:Sehwag2021Proxy_R18'    # B2-verified base ([0,1], clean 0.8459)
EPOCHS=15; SEED=0                          # LOCKED config: lr 0.005, warmup 2 (script defaults)
C1K ='configs/eval/audit_cifar10_preactrn18_multinorm_v3A_testfinal.yaml'
C10K='results/eval/union_bench/_config/audit_v3A_test_10k.yaml'
import os; os.makedirs(OUT, exist_ok=True)
print('base', BASE, '| epochs', EPOCHS, '| lr 0.005 | warmup 2 | full fine-tune | seed', SEED)

## Base context — audit the UN-fine-tuned base @1k no-Square (trajectory start)

Gives a 3-point trajectory **base → M0_ft → M1_ft**, so the fine-tune's effect (and CLAMP's Δ)
is read against where the base started, not in a vacuum.

In [ ]:
import subprocess, sys, os, json, torch
RB='Sehwag2021Proxy_R18'; CK=f'{REPO}/models/cifar10/Linf/{RB}.pt'
if not os.path.exists(CK):                                  # ensure the base is downloaded
    _o=torch.load; torch.load=lambda *a,**k:_o(*a,**{**k,'weights_only':False})
    try:
        from robustbench.utils import load_model
        load_model(model_name=RB,dataset='cifar10',threat_model='Linf',model_dir=f'{REPO}/models')
    finally: torch.load=_o
out=f'{DRIVE}/union_bench/rb_base_nosq/eval.json'; os.makedirs(os.path.dirname(out),exist_ok=True)
if not os.path.exists(out):
    subprocess.run([sys.executable,f'{REPO}/scripts/dev/union_bench_eval.py','--config',C1K,'--checkpoint',CK,
        '--arch','robustbench','--run-id',f'{RB}_base_1k','--checkpoint-role','published',
        '--out',out,'--export-masks','--bs','128','--skip-square'],cwd=REPO,env=dict(os.environ,ATTACKDRO_ROOT=REPO))
db=json.load(open(out))
print(f"BASE (no fine-tune) @1k no-Square:  clean={db['clean_acc']:.4f}  union={db['full_audit_union']:.4f}")

## B3 — fine-tune the matched pair (M1_ft = CLAMP · M0_ft = no term)

~1 h/arm. Re-running skips a finished arm (train.json present). If Colab drops mid-arm, just
re-run — the arm restarts (fine-tune is short; not mid-epoch resume-safe).

In [ ]:
import subprocess, sys, os
def finetune(arm):
    outdir=f'{OUT}/{arm}'
    if os.path.exists(f'{outdir}/train.json'):
        import json; print(f'SKIP ft-{arm} (done, best_valWU={json.load(open(outdir+"/train.json"))["best_val_worst_union"]:.4f})'); return
    cmd=[sys.executable,f'{REPO}/scripts/dev/finetune_msd_clamp.py','--arm',arm,'--base',BASE,
         '--epochs',str(EPOCHS),'--seed',str(SEED),'--outdir',outdir]
    print('$',' '.join(cmd),'\n')
    env=dict(os.environ, ATTACKDRO_ROOT=REPO, C5_NUM_WORKERS='2')
    p=subprocess.Popen(cmd,cwd=REPO,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in p.stdout: print(line,end='')
    print('[exit',p.wait(),']')
finetune('clamp')       # M1_ft
finetune('none')        # M0_ft

## B4 — tiered eval: no-Square @1k (fast Δ), union + clean

Audits the fine-tuned RB arms through OUR harness (`--arch robustbench --rb-name` reconstructs
the arch + loads the fine-tuned weights; `--skip-square` = fast tier). Δ union + clean.
If promising → run the 10k cell below.

In [ ]:
import subprocess, sys, os, json
RB='Sehwag2021Proxy_R18'
def audit(arm, scale, skip_square):
    ck=f'{OUT}/{arm}/ckpt/val_best.pt'
    tag=f'ft_{arm}'+('_nosq' if skip_square else '')
    out=f'{DRIVE}/union_bench/{tag}/'+('' if scale=='1k' else '10k/')+'eval.json'
    if os.path.exists(out): print('SKIP',tag,scale); return json.load(open(out))
    if not os.path.exists(ck): print('missing ckpt',ck); return None
    os.makedirs(os.path.dirname(out),exist_ok=True)
    cfg=C1K if scale=='1k' else C10K
    cmd=[sys.executable,f'{REPO}/scripts/dev/union_bench_eval.py','--config',cfg,'--checkpoint',ck,
         '--arch','robustbench','--rb-name',RB,'--run-id',f'{tag}_{scale}','--checkpoint-role','finetuned',
         '--out',out,'--export-masks','--bs','128']+(['--skip-square'] if skip_square else [])
    r=subprocess.run(cmd,cwd=REPO,env=dict(os.environ,ATTACKDRO_ROOT=REPO),capture_output=True,text=True)
    if not os.path.exists(out): print(r.stdout[-1200:]); print(r.stderr[-1500:]); return None
    return json.load(open(out))

print('=== FAST TIER: no-Square @1k — trajectory base -> M0_ft -> M1_ft ===')
res={}
for arm in ['clamp','none']:
    d=audit(arm,'1k',skip_square=True)
    if d: res[arm]=d
# 3-point trajectory
try:
    import json as _j
    db=_j.load(open(f'{DRIVE}/union_bench/rb_base_nosq/eval.json'))
    print(f"  {'arm':10} {'clean':>8} {'union(no-Sq)':>13}")
    print(f"  {'base':10} {db['clean_acc']:8.4f} {db['full_audit_union']:13.4f}")
    for arm in ['none','clamp']:
        if arm in res: print(f"  {'M0_ft' if arm=='none' else 'M1_ft':10} {res[arm]['clean_acc']:8.4f} {res[arm]['full_audit_union']:13.4f}")
except Exception as e: print('  base context missing:', e)
if 'clamp' in res and 'none' in res:
    du=res['clamp']['full_audit_union']-res['none']['full_audit_union']
    dc=res['clamp']['clean_acc']-res['none']['clean_acc']
    print(f"\n  Δ (M1_ft - M0_ft) @1k no-Square:  union {du:+.4f}  | clean {dc:+.4f}")
    print('  -> promising (union up)? run the 12-AA @10k cell.' if du>0 else '  -> flat/negative; report as-is (pre-registered).')

## B4b — full 12-AA @10k (only if the fast tier is promising)

In [ ]:
# run each arm's full 12-AA @10k, then paired-style Δ (union + clean)
for arm in ['clamp','none']:
    d=audit(arm,'10k',skip_square=False)
    if d: print(f"ft_{arm} 10k: clean={d['clean_acc']:.4f} union={d['full_audit_union']:.4f}")
import json, os
try:
    c=json.load(open(f'{DRIVE}/union_bench/ft_clamp/10k/eval.json'))
    n=json.load(open(f'{DRIVE}/union_bench/ft_none/10k/eval.json'))
    print(f"\nΔ (clamp - none) @10k FULL:  union {c['full_audit_union']-n['full_audit_union']:+.4f}"
          f"  | clean {c['clean_acc']-n['clean_acc']:+.4f}")
except Exception as e: print('10k not both present yet:', e)